# Report Figures

Draws all 35 README figures from the final dataset, each as an opaque PNG plus a `-transparent.png` twin, saved to `figures/`. The notebook runs last in the pipeline, after notebook 8 (`combine_access_score_network_threshold`) has written `data/final/sal_combined_access.geojson` and `data/final/pharmacies.geojson`. Before drawing anything it checks that the final dataset still reproduces the project's headline numbers and refuses to run otherwise.

The notebook has two sections:

- **Exploratory Data Analysis:** the 16 context figures that describe the study area and the data.
- **Results:** sections A–H of the exploration notebook, which test the report's claims and provide the numbers for the write-up.

Figures are styled by `dair_style.py`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()          # repository root; notebooks run from notebooks/
DATA = ROOT / "data"
RAW = DATA / "raw"
INTERMEDIATE = DATA / "intermediate"
FINAL = DATA / "final"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import dair_style as ds

IN_SAL_COMBINED = ds.SAL_SOURCE
IN_PHARMACIES_GEOJSON = ds.PHARMACIES_SOURCE
IN_SOUTH_AFRICA = ds.PROVINCE_OUTLINES_SOURCE


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm, ListedColormap, Normalize, TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

from dair_style import (
    COUNT_CMAP, DIST_BINS, DIST_COLORS, DIST_LABELS, HEAVY_BEIGE, INK, MISSING_COLOR, NO_DWELLINGS,
    OFF_WHITE, POP, PROV_COLORS, PROVINCES, RACE_COLORS, RACE_COLS,
    SETTLEMENT_COLORS, SETTLEMENT_MAP, SETTLEMENT_ORDER, SHARE_CMAP,
    TYPE6_COLORS, VERMILLION, WEALTH_LABELS, WEALTH_ORDER,
    bar_labels, contrast_text, heat_matrix, legend_below, plot_categories, ramp_colors, save_fig,
    style_bar_axis, title,
)

# --- Configuration -----------------------------------------------------------
HEADLINE = 5_000_000               # "approximately 5 million people"


# --- Helpers -----------------------------------------------------------------
def millions(x, _=None):
    return f"{x / 1e6:.1f}M"


def share_table(frame, by, flag, min_pop=0):
    """Population-weighted share of each group where `flag` is True."""
    den = frame.groupby(by)[POP].sum()
    num = frame.loc[frame[flag]].groupby(by)[POP].sum().reindex(den.index, fill_value=0)
    return num / den.where(den > min_pop)


def weighted_median(values, weights):
    v, w = np.asarray(values, float), np.asarray(weights, float)
    ok = ~np.isnan(v) & ~np.isnan(w) & (w > 0)
    v, w = v[ok], w[ok]
    if w.sum() == 0:
        return np.nan
    order = np.argsort(v)
    v, cum = v[order], np.cumsum(w[order])
    return v[np.searchsorted(cum, cum[-1] / 2)]


In [ ]:
def settlement_of(ea_type):
    """Map raw EA_TYPE values to Settlement Types."""
    return np.where(
        ea_type.astype(str).str.endswith("_*"),
        ds.NO_DWELLINGS,
        ea_type.map(ds.SETTLEMENT_MAP),
    )


def load_data():
    """Load SALs, pharmacies, and province outlines, all in PROJECTED_CRS."""
    sal = gpd.read_file(ds.SAL_SOURCE).to_crs(ds.PROJECTED_CRS)
    sal["settlement"] = settlement_of(sal["EA_TYPE"])

    # PROVINCE on the pharmacy file is inconsistent; take the province from the
    # SAL the point falls inside.
    pharmacies = gpd.read_file(ds.PHARMACIES_SOURCE).to_crs(ds.PROJECTED_CRS)
    pharmacies = gpd.sjoin(pharmacies, sal[["PR_NAME", "geometry"]], predicate="within", how="left")

    outlines = gpd.read_file(ds.PROVINCE_OUTLINES_SOURCE).to_crs(ds.PROJECTED_CRS)
    return sal, pharmacies, outlines


In [ ]:
sal, pharmacies, outlines = load_data()

required = ["EA_CODE", "PR_NAME", "EA_TYPE", "EA_GTYPE", "econ_stat", POP,
            "wlk_type", "drv_type", "wlk_type3", "drv_type3", "wlk_ai", "drv_ai",
            "wlk_k1_km", "drv_k1_km", "euc_k1_km", "wk1_gt3km", "wk1_gt5km", "dk1_gt10km",
            "wlk_snapfl", "drv_snapfl", *RACE_COLS]
missing = [c for c in required if c not in sal.columns]
assert not missing, f"Missing columns (is this the v2 SAL dataset?): {missing}"
print(f"{len(sal):,} SALs loaded")


In [ ]:
# Headline check: refuse to draw figures unless the final dataset reproduces
# the project's headline numbers (spec Appendix D).
failures = []

n_rows = len(sal)
print(f"SAL rows: {n_rows:,}")
if n_rows != 38_380:
    failures.append(f"row count {n_rows:,} != 38,380")

province_rows = sal["PR_NAME"].value_counts()
gauteng_rows = province_rows.get("Gauteng", 0)
kwazulu_natal_rows = province_rows.get("KwaZulu-Natal", 0)
print(f"Gauteng rows: {gauteng_rows:,}")
print(f"KwaZulu-Natal rows: {kwazulu_natal_rows:,}")
if gauteng_rows != 20_850:
    failures.append(f"Gauteng rows {gauteng_rows:,} != 20,850")
if kwazulu_natal_rows != 17_530:
    failures.append(f"KwaZulu-Natal rows {kwazulu_natal_rows:,} != 17,530")

total_pop = sal[ds.POP].sum()
print(f"Modeled Population: {total_pop / 1e6:.2f}M")
if round(total_pop / 1e6, 1) != 27.5:
    failures.append(f"Modeled Population {total_pop / 1e6:.2f}M does not round to 27.5M")

walk_desert_pop = sal.loc[sal["wlk_type"].eq("Pharmacy Desert"), ds.POP].sum()
print(f"Walking Pharmacy Desert population: {walk_desert_pop / 1e6:.2f}M")
if round(walk_desert_pop / 1e6, 2) != 10.20:
    failures.append(f"walking Pharmacy Desert population {walk_desert_pop / 1e6:.2f}M does not round to 10.20M")

drive_desert_pop = sal.loc[sal["drv_type"].eq("Pharmacy Desert"), ds.POP].sum()
print(f"Driving Pharmacy Desert population: {drive_desert_pop / 1e6:.2f}M")
if round(drive_desert_pop / 1e6, 2) != 4.63:
    failures.append(f"driving Pharmacy Desert population {drive_desert_pop / 1e6:.2f}M does not round to 4.63M")

print(f"Pharmacies: {len(pharmacies):,}")
if len(pharmacies) != 2_152:
    failures.append(f"pharmacy count {len(pharmacies):,} != 2,152")

if failures:
    raise ValueError("Headline check failed: " + "; ".join(failures))
print("Headline checks passed")


In [ ]:
gdf = sal.copy()

# Derived columns used throughout
FLAG_COLS = ["wk1_gt3km", "wk1_gt5km", "dk1_gt10km", "wlk_snapfl", "drv_snapfl"]
gdf[FLAG_COLS] = gdf[FLAG_COLS].astype(bool)  # guards against 0/1 or object dtypes
gdf["settlement"] = np.where(gdf["EA_TYPE"].str.endswith("_*", na=False),
                             NO_DWELLINGS, gdf["EA_TYPE"].map(SETTLEMENT_MAP))

# Older exports store the 6-category labels in sentence case ("Pharmacy desert");
# normalize to the Title Case labels used in TYPE6_COLORS.
TYPE6_LOOKUP = {k.lower(): k for k in TYPE6_COLORS}
for col in ["wlk_type", "drv_type"]:
    normalized = gdf[col].str.lower().map(TYPE6_LOOKUP)
    unknown = set(gdf.loc[normalized.isna() & gdf[col].notna(), col])
    assert not unknown, f"Unexpected {col} labels: {unknown}"
    gdf[col] = normalized

gdf["walk_desert"] = gdf["wlk_type"].eq("Pharmacy Desert")
gdf["drive_desert"] = gdf["drv_type"].eq("Pharmacy Desert")

race_total = gdf[list(RACE_COLS)].sum(axis=1, min_count=1)
has_race = race_total > 0
gdf["pct_blk_afr"] = (gdf["Black_Afr"] / race_total).where(has_race)
majority = gdf[list(RACE_COLS)].fillna(-1).idxmax(axis=1).map(RACE_COLS)
gdf["majority_group"] = majority.where(has_race)
# Majority Population Group: the largest 2011 group (a plurality); ties go to the first group in RACE_COLS order.

for mode, col in [("walk", "wlk_k1_km"), ("drive", "drv_k1_km")]:
    classes = pd.cut(gdf[col], DIST_BINS, labels=DIST_LABELS, right=False)
    gdf[f"{mode}_dist_class"] = classes.cat.add_categories("Unreachable").fillna("Unreachable").astype(str)

df = pd.DataFrame(gdf.drop(columns="geometry"))

summary = df.groupby("PR_NAME").agg(**{"SALs": ("EA_CODE", "size"), "Modeled Population (2023)": (POP, "sum")})
summary.loc["Both Provinces"] = summary.sum()
display(summary.assign(**{"Modeled Population (2023)": summary["Modeled Population (2023)"].map(millions)}))
print("\nSettlement Types:", df["settlement"].value_counts().to_dict())

## Exploratory Data Analysis



### Study Area: Gauteng and KwaZulu-Natal


In [ ]:
fig, (ax_za, ax_gp, ax_kz) = plt.subplots(
        1, 3, figsize=(14, 5.6), width_ratios=[1, 0.9, 1.1], layout="constrained")
outlines.plot(ax=ax_za, color=ds.CARTON_BEIGE, edgecolor=ds.HEAVY_BEIGE, linewidth=0.6)
for prov in ds.PROVINCES:
        outlines[outlines["name"].eq(prov)].plot(
            ax=ax_za, color=ds.PROV_COLORS[prov], edgecolor=ds.BROWN, linewidth=0.6)
ax_za.set_ylim(6_050_000, None)
ax_za.set_aspect("equal")
ax_za.set_axis_off()
ax_za.set_title("South Africa")
for ax, prov in zip((ax_gp, ax_kz), ds.PROVINCES):
        prov_sal = sal[sal["PR_NAME"].eq(prov)]
        prov_sal.plot(ax=ax, color=ds.PROV_COLORS[prov], edgecolor=ds.OFF_WHITE, linewidth=0.05)
        ax.set_aspect("equal")
        ax.set_axis_off()
        ds.add_scalebar(ax)
        pop = prov_sal[ds.POP].sum()
        area = prov_sal["area_km2"].sum()
        stats = (f"SALs: {len(prov_sal):,}\n"
                 f"Modeled Population (2023): {pop / 1e6:.1f}M\n"
                 f"Area: {area:,.0f} km\u00b2\n"
                 f"Density: {pop / area:,.0f} per km\u00b2")
        ax.text(0.98, 0.02, stats, transform=ax.transAxes, ha="right", va="bottom", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.4", facecolor=ds.OFF_WHITE, edgecolor=ds.HEAVY_BEIGE))
ds.title(fig, "Study Area: Gauteng and KwaZulu-Natal")
ds.save_fig("study-area-locator-map", fig)


### SALs and Pharmacies by Province


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5), width_ratios=[1, 1.6], layout="constrained")
for ax, prov in zip(axes, ds.PROVINCES):
        prov_sal = sal[sal["PR_NAME"].eq(prov)]
        prov_pharm = pharmacies[pharmacies["PR_NAME"].eq(prov)]
        prov_sal.plot(ax=ax, color=ds.HEAVY_BEIGE, edgecolor=ds.CARTON_BEIGE, linewidth=0.1)
        prov_pharm.plot(ax=ax, color=ds.NAVY, markersize=3, edgecolor="none")
        ax.set_title(f"{prov}: {len(prov_sal):,} SALs, {len(prov_pharm):,} Pharmacies")
        ax.set_aspect("equal")
        ax.set_axis_off()
        ds.add_scalebar(ax)
ds.title(fig, "SALs and Pharmacies by Province")
ds.save_fig("2sfca-province-preview", fig)


### Population Density by Settlement Type


In [ ]:
def settlement_density(sal):
    """Population density per province and Settlement Type, No Dwellings excluded."""
    frame = sal[sal["settlement"].ne(ds.NO_DWELLINGS)]
    grouped = frame.groupby(["PR_NAME", "settlement"], observed=True).agg(
        pop=(ds.POP, "sum"), area=("area_km2", "sum"))
    density = (grouped["pop"] / grouped["area"]).rename("density").to_frame()
    return density

density = settlement_density(sal)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True, layout="constrained")
for ax, prov in zip(axes, ds.PROVINCES):
        order = [s for s in ds.SETTLEMENT_ORDER if s != ds.NO_DWELLINGS]
        prov_density = density.loc[prov].reindex(order).dropna()
        prov_density = prov_density[prov_density["density"] > 0]
        ax.barh(np.arange(len(prov_density)), prov_density["density"], height=0.7,
                color=[ds.SETTLEMENT_COLORS[s] for s in prov_density.index])
        ax.bar_label(ax.containers[0], labels=[f" {v:,.0f}" for v in prov_density["density"]],
                     fontsize=9, padding=2)
        ax.set_yticks(np.arange(len(prov_density)), list(prov_density.index))
        ax.invert_yaxis()
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
        ds.style_bar_axis(ax, "x")
        ax.set_title(prov)
        ax.set_xlabel("Residents per km\u00b2 (Modeled Population, 2023)")
ds.title(fig, "Population Density by Settlement Type")
ds.save_fig("settlement-type-density", fig)


### Walking vs. Driving Access Score Terciles


In [ ]:
def score_terciles(score):
    """Zero scores stay 0; nonzero scores get tercile labels 0, 1, 2 together."""
    terciles = pd.Series(0, index=score.index, dtype=int)
    positive = score[score > 0]
    terciles.loc[positive.index] = pd.qcut(positive, 3, labels=[0, 1, 2]).astype(int)
    return terciles

def bivariate_counts(sal):
    """SAL counts per (walk_class, drive_class) over both provinces together."""
    walk = score_terciles(sal["wlk_ai_log"])
    drive = score_terciles(sal["drv_ai_log"])
    return {key: int((walk.eq(key[0]) & drive.eq(key[1])).sum()) for key in ds.BIVARIATE_COLORS}

frame = sal.copy()
frame["walk_class"] = score_terciles(frame["wlk_ai_log"])
frame["drive_class"] = score_terciles(frame["drv_ai_log"])
counts = bivariate_counts(frame)
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5), width_ratios=[1, 1.6], layout="constrained")
for ax, prov in zip(axes, ds.PROVINCES):
        prov_sal = frame[frame["PR_NAME"].eq(prov)]
        for (walk_class, drive_class), color in ds.BIVARIATE_COLORS.items():
            sub = prov_sal[prov_sal["walk_class"].eq(walk_class) & prov_sal["drive_class"].eq(drive_class)]
            if len(sub):
                sub.plot(ax=ax, color=color, linewidth=0)
        ax.set_aspect("equal")
        ax.set_axis_off()
        ax.set_title(prov)
        ds.add_scalebar(ax)
inset = fig.add_axes([0.86, 0.08, 0.11, 0.2])
inset.set_in_layout(False)
for (walk_class, drive_class), color in ds.BIVARIATE_COLORS.items():
        inset.add_patch(Rectangle((drive_class, walk_class), 1, 1, facecolor=color,
                                  edgecolor=ds.OFF_WHITE, linewidth=1.5))
        inset.text(drive_class + 0.5, walk_class + 0.5, f"{counts[(walk_class, drive_class)]:,}",
                   ha="center", va="center", fontsize=7, color=ds.contrast_text(color))
inset.set_xlim(-0.05, 3.05)
inset.set_ylim(-0.05, 3.05)
inset.set_aspect("equal")
inset.set_xlabel("Driving Access \u2192", fontweight="medium")
inset.set_ylabel("Walking Access \u2192", fontweight="medium")
inset.set_xticks([])
inset.set_yticks([])
for spine in inset.spines.values():
        spine.set_visible(False)
ds.title(fig, "Walking vs. Driving Access Score (Terciles of Nonzero Scores)")
ds.save_fig("2sfca-bivariate-walk-drive", fig)


### Walking Access Score: Johannesburg and Pretoria


In [ ]:
def _walk_score_city(sal, pharmacies, city, cities_to_label, name, fig_title, legend_text_color=ds.INK):
    x0, x1, y0, y1 = ds.CITY_EXTENTS[city]
    sub = sal.cx[x0:x1, y0:y1]
    pts = pharmacies.cx[x0:x1, y0:y1]
    norm = Normalize(vmin=sub["wlk_ai_log"].quantile(0.02), vmax=sub["wlk_ai_log"].quantile(0.98))
    fig, ax = plt.subplots(figsize=(10, 10 * (y1 - y0) / (x1 - x0)), layout="constrained")
    sub.plot(ax=ax, column="wlk_ai_log", cmap=ds.ACCESS_CMAP, norm=norm,
             edgecolor=ds.OFF_WHITE, linewidth=0.05)
    pts.plot(ax=ax, color=ds.OFF_WHITE, markersize=8, edgecolor=ds.NAVY, linewidth=0.3, zorder=5)
    ds.set_extent(ax, ds.CITY_EXTENTS[city])
    ds.label_cities(ax, cities_to_label, color=ds.VERMILLION, size=44)
    ds.add_scalebar(ax)
    colorbar = fig.colorbar(ScalarMappable(norm=norm, cmap=ds.ACCESS_CMAP), ax=ax,
                            fraction=0.035, pad=0.02, extend="max")
    colorbar.outline.set_visible(False)
    colorbar.set_label("Walking Access Score, ln(1 + Score)", fontweight="medium")
    legend = ax.legend(handles=[Line2D([], [], marker="o", ls="", markerfacecolor=ds.OFF_WHITE,
                              markeredgecolor=ds.NAVY, label="Pharmacy")], loc="upper right")
    legend.get_texts()[0].set_color(legend_text_color)
    ds.title(fig, fig_title)
    return ds.save_fig(name, fig)

_walk_score_city(sal, pharmacies, "Johannesburg",
                            ["Johannesburg", "Pretoria", "Olievenhoutbosch"],
                            "2sfca-walk-score-johannesburg",
                            "Walking Access Score: Johannesburg and Pretoria",
                            legend_text_color=ds.OFF_WHITE)


### Walking Access Score: Durban


In [ ]:
_walk_score_city(sal, pharmacies, "Durban",
                            ["Durban", "KwaMashu"],
                            "2sfca-walk-score-durban",
                            "Walking Access Score: Durban")


### Distance to the Nearest Pharmacy: Gauteng


In [ ]:
DISTANCE_BINS_KM = [0, 1, 2, 3, 5, 10, 15, 25, 50]

DISTANCE_LEGEND_KEYS = ["0-1 km", "1-2 km", "2-3 km", "3-5 km", "5-10 km",
                        "10-15 km", "15-25 km", "25+ km", "Unreachable Within 50 km"]

def _distance_panels(sal, province, k, name):
    prov_sal = sal[sal["PR_NAME"].eq(province)]
    fig, axes = plt.subplots(1, 3, figsize=(16, 6.5), layout="constrained")
    cmap = ListedColormap(ds.ramp_colors(ds.DISTANCE_CMAP, 8))
    norm = BoundaryNorm(DISTANCE_BINS_KM, ncolors=8)
    panels = [("Straight-Line", f"euc_k{k}_km"),
              ("Walking", f"wlk_k{k}_km"),
              ("Driving", f"drv_k{k}_km")]
    for ax, (panel_title, col) in zip(axes, panels):
        missing = prov_sal[prov_sal[col].isna()]
        if len(missing):
            missing.plot(ax=ax, color=ds.UNREACHABLE_COLOR, linewidth=0)
        present = prov_sal[prov_sal[col].notna()].copy()
        present["dist"] = present[col].clip(upper=50)
        present.plot(ax=ax, column="dist", cmap=cmap, norm=norm, linewidth=0)
        ax.set_aspect("equal")
        ax.set_axis_off()
        ax.set_title(panel_title)
    ds.add_scalebar(axes[0])

    legend_colors = dict(zip(DISTANCE_LEGEND_KEYS[:8], ds.ramp_colors(ds.DISTANCE_CMAP, 8)))
    legend_colors[DISTANCE_LEGEND_KEYS[8]] = ds.UNREACHABLE_COLOR
    ds.legend_below(fig, legend_colors, ncol=9, title="Distance to Pharmacy")
    fig_title = (f"{province}: Distance to the Nearest Pharmacy" if k == 1
                 else f"{province}: Distance to the Third-Nearest Pharmacy")
    ds.title(fig, fig_title)
    return ds.save_fig(name, fig)

_distance_panels(sal, "Gauteng", 1, "distance-k1-gauteng")


### Distance to the Third-Nearest Pharmacy: Gauteng


In [ ]:
_distance_panels(sal, "Gauteng", 3, "distance-k3-gauteng")


### Distance to the Nearest Pharmacy: KwaZulu-Natal


In [ ]:
_distance_panels(sal, "KwaZulu-Natal", 1, "distance-k1-kwazulu-natal")


### Distance to the Third-Nearest Pharmacy: KwaZulu-Natal


In [ ]:
_distance_panels(sal, "KwaZulu-Natal", 3, "distance-k3-kwazulu-natal")


### Population Group Composition, 2011 Census


In [ ]:
GROUP_COLS = list(ds.RACE_COLS)

def population_group_shares(sal):
    """Per province: each Population Group's share of the 2011 group total (SALs with data)."""
    rows = []
    for prov in ds.PROVINCES:
        prov_sal = sal[sal["PR_NAME"].eq(prov)]
        group_total = prov_sal[GROUP_COLS].sum(axis=1)
        prov_sal = prov_sal[group_total > 0]
        sums = prov_sal[GROUP_COLS].sum()
        rows.append((prov, float(sums.sum()), sums / sums.sum()))
    return rows

rows = population_group_shares(sal)
fig, ax = plt.subplots(figsize=(12, 3.6), layout="constrained")
for y, (prov, total, shares) in enumerate(rows):
        left = 0.0
        for col in GROUP_COLS:
            label = ds.RACE_COLS[col]
            share = float(shares[col])
            color = ds.RACE_COLORS[label]
            ax.barh(y, share, left=left, height=0.6, color=color, edgecolor=ds.OFF_WHITE, linewidth=1)
            if share >= 0.04:
                ax.text(left + share / 2, y, f"{share:.0%}", ha="center", va="center",
                        fontsize=9, fontweight="semibold", color=ds.contrast_text(color))
            left += share
ax.set_yticks(range(len(rows)), [f"{prov}\n({total / 1e6:.1f}M, 2011)" for prov, total, _ in rows])
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1))
ax.set_xlim(0, 1)
ax.spines["left"].set_visible(False)
ds.legend_below(fig, ds.RACE_COLORS, ncol=5)
ds.title(fig, "Population Group Composition, 2011 Census")
ds.save_fig("population-group-composition", fig)


### Modeled Population per SAL, 2023


In [ ]:
edges = np.unique(sal[ds.POP].quantile([0, 0.2, 0.4, 0.6, 0.8, 1]).to_numpy())
codes = pd.cut(sal[ds.POP], edges, include_lowest=True).cat.codes
colors = ds.ramp_colors(ds.COUNT_CMAP, len(edges) - 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5), width_ratios=[1, 1.6], layout="constrained")
for ax, prov in zip(axes, ds.PROVINCES):
        prov_sal = sal[sal["PR_NAME"].eq(prov)]
        prov_codes = codes.loc[prov_sal.index]
        for code, color in enumerate(colors):
            sub = prov_sal[prov_codes.eq(code)]
            if len(sub):
                sub.plot(ax=ax, color=color, linewidth=0)
        ax.set_aspect("equal")
        ax.set_axis_off()
        ds.add_scalebar(ax)
labels = [f"0-{edges[1]:,.0f}"] + [f"{edges[i]:,.0f}-{edges[i + 1]:,.0f}" for i in range(1, len(edges) - 1)]
ds.legend_below(fig, dict(zip(labels, colors)), title="Residents per SAL (Modeled Population, 2023)")
ds.title(fig, "Modeled Population per SAL, 2023")
ds.save_fig("population-2023-estimate", fig)


### Population Density per SAL, 2023


In [ ]:
DENSITY_BINS = [0, 100, 500, 1_000, 2_500, 5_000, 10_000, 20_000, np.inf]

DENSITY_LABELS = ["< 100", "100-500", "500-1,000", "1,000-2,500",
                  "2,500-5,000", "5,000-10,000", "10,000-20,000", "20,000+"]

def density_codes(sal):
    """Class index per SAL for the fixed density bins."""
    density = sal[ds.POP] / sal["area_km2"]
    return pd.cut(density, DENSITY_BINS, right=False).cat.codes

codes = density_codes(sal)
colors = ds.ramp_colors(ds.COUNT_CMAP, 8)
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5), width_ratios=[1, 1.6], layout="constrained")
for ax, prov in zip(axes, ds.PROVINCES):
        prov_sal = sal[sal["PR_NAME"].eq(prov)]
        prov_codes = codes.loc[prov_sal.index]
        for code, color in enumerate(colors):
            sub = prov_sal[prov_codes.eq(code)]
            if len(sub):
                sub.plot(ax=ax, color=color, linewidth=0)
        ax.set_aspect("equal")
        ax.set_axis_off()
        ds.add_scalebar(ax)
ds.legend_below(fig, dict(zip(DENSITY_LABELS, colors)), title="Residents per km\u00b2 (Modeled Population, 2023)")
ds.title(fig, "Population Density per SAL, 2023")
ds.save_fig("population-density-2023", fig)


### Population Density: Gauteng and the City of Johannesburg


In [ ]:
codes = density_codes(sal)
colors = ds.ramp_colors(ds.COUNT_CMAP, 8)
gauteng = sal[sal["PR_NAME"].eq("Gauteng")]
joburg = gauteng[gauteng["MN_NAME"].eq("City of Johannesburg")]
fig, (ax_gp, ax_jb) = plt.subplots(1, 2, figsize=(14, 7), width_ratios=[1, 1.6], layout="constrained")
for code, color in enumerate(colors):
        sub = gauteng[codes.loc[gauteng.index].eq(code)]
        if len(sub):
            sub.plot(ax=ax_gp, color=color, linewidth=0)
ax_gp.set_title("Gauteng")
x0, y0, x1, y1 = joburg.total_bounds
pad_x = 0.02 * (x1 - x0)
pad_y = 0.02 * (y1 - y0)
ax_gp.add_patch(Rectangle((x0 - pad_x, y0 - pad_y), (x1 - x0) + 2 * pad_x, (y1 - y0) + 2 * pad_y,
                              edgecolor=ds.VERMILLION, facecolor="none", linewidth=2))
ax_gp.set_xlim(gauteng.total_bounds[0], gauteng.total_bounds[2])
ax_gp.set_ylim(gauteng.total_bounds[1], gauteng.total_bounds[3])
ax_gp.set_aspect("equal")
ax_gp.set_axis_off()
ds.add_scalebar(ax_gp)
for code, color in enumerate(colors):
        sub = joburg[codes.loc[joburg.index].eq(code)]
        if len(sub):
            sub.plot(ax=ax_jb, color=color, edgecolor=ds.OFF_WHITE, linewidth=0.05)
ax_jb.set_xlim(x0 - pad_x, x1 + pad_x)
ax_jb.set_ylim(y0 - pad_y, y1 + pad_y)
ax_jb.set_aspect("equal")
ax_jb.set_axis_off()
ax_jb.set_title("City of Johannesburg")
ds.label_cities(ax_jb, ["Johannesburg"], color=ds.VERMILLION)
ds.add_scalebar(ax_jb)
ds.legend_below(fig, dict(zip(DENSITY_LABELS, colors)), title="Residents per km\u00b2 (Modeled Population, 2023)")
ds.title(fig, "Population Density, Gauteng and the City of Johannesburg, 2023")
ds.save_fig("johannesburg-zoom", fig)


### Annual Population Growth Rate per SAL, 2011–2023


In [ ]:
CITIES_PER_PROVINCE = {
    "Gauteng": ["Johannesburg", "Pretoria", "Olievenhoutbosch"],
    "KwaZulu-Natal": ["Durban", "KwaMashu"],
}

low, high = sal["growth_rt"].quantile([0.02, 0.98])
norm = TwoSlopeNorm(vmin=low, vcenter=0, vmax=high)
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5), width_ratios=[1, 1.6], layout="constrained")
for ax, prov in zip(axes, ds.PROVINCES):
        prov_sal = sal[sal["PR_NAME"].eq(prov)]
        no_growth = prov_sal[prov_sal["growth_rt"].isna()]
        if len(no_growth):
            no_growth.plot(ax=ax, facecolor=ds.PAPER_BEIGE, edgecolor=ds.HEAVY_BEIGE,
                           hatch="////", linewidth=0)
        with_growth = prov_sal[prov_sal["growth_rt"].notna()]
        with_growth.plot(ax=ax, column="growth_rt", cmap=ds.GROWTH_CMAP, norm=norm, linewidth=0)
        ax.set_aspect("equal")
        ax.set_axis_off()
        ds.add_scalebar(ax)
        ds.label_cities(ax, CITIES_PER_PROVINCE[prov], color=ds.BROWN)
colorbar = fig.colorbar(ScalarMappable(norm=norm, cmap=ds.GROWTH_CMAP), ax=axes,
                            fraction=0.025, pad=0.02, extend="both")
colorbar.outline.set_visible(False)
colorbar.ax.yaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=0))
colorbar.set_label("Annual Growth Rate, 2011\u20132023", fontweight="medium")
fig.legend(handles=[Patch(facecolor=ds.PAPER_BEIGE, edgecolor=ds.HEAVY_BEIGE, hatch="////",
                              label="No 2011 Population")], loc="outside lower left")
ds.title(fig, "Annual Population Growth Rate per SAL, 2011\u20132023")
ds.save_fig("growth-rate-map", fig)


### Registered Pharmacies, Gauteng and KwaZulu-Natal


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5), width_ratios=[1, 1.6], layout="constrained")
for ax, prov in zip(axes, ds.PROVINCES):
        prov_sal = sal[sal["PR_NAME"].eq(prov)]
        prov_sal.plot(ax=ax, color=ds.CARTON_BEIGE, linewidth=0)
        prov_pharm = pharmacies[pharmacies["PR_NAME"].eq(prov)]
        prov_pharm.plot(ax=ax, color=ds.NAVY, markersize=4, edgecolor="none")
        ax.set_title(f"{prov}: {len(prov_pharm):,} Pharmacies")
        ax.set_aspect("equal")
        ax.set_axis_off()
        ds.add_scalebar(ax)
        ds.label_cities(ax, CITIES_PER_PROVINCE[prov], color=ds.VERMILLION)
ds.title(fig, "Registered Pharmacies, Gauteng and KwaZulu-Natal")
ds.save_fig("pharmacy-locations", fig)


## Results



## A. The Headline: "Approximately 5 Million People Live in a Pharmacy Desert"

"Pharmacy Desert" has no single definition in the data, so this compares every reasonable one. The Access Typology's Pharmacy Desert means **no pharmacy within the 2SFCA catchment and the nearest one beyond the distance threshold** (3 km walking, 10 km driving).


In [ ]:
# A1. Population in a pharmacy desert under each definition
definitions = {
    "Walk Typology: Pharmacy Desert (Score = 0, Nearest > 3 km)": df["walk_desert"],
    "Walk: Nearest > 3 km (Reachable SALs Only)": df["wk1_gt3km"],
    "Walk: Nearest > 3 km or No Path Within 50 km": df["wlk_k1_km"].gt(3) | df["wlk_k1_km"].isna(),
    "Walk: Score = 0 (None in Catchment)": df["wlk_ai"].eq(0),
    "Walk 3-Tier: Score = 0 or Bottom Tercile": df["wlk_type3"].eq("Pharmacy Desert"),
    "Drive Typology: Pharmacy Desert (Score = 0, Nearest > 10 km)": df["drive_desert"],
    "Drive: Nearest > 10 km (Reachable SALs Only)": df["dk1_gt10km"],
    # Unreachable SALs that are also > 10 km straight-line are likely genuine;
    # unreachable SALs near a pharmacy are more likely OSM network gaps.
    "Drive: > 10 km, Plus Unreachable SALs > 10 km Straight-Line":
        df["drv_k1_km"].gt(10) | (df["drv_k1_km"].isna() & df["euc_k1_km"].gt(10)),
    "Drive: Nearest > 10 km or No Path Within 50 km": df["drv_k1_km"].gt(10) | df["drv_k1_km"].isna(),
}
# Note: the exceedance flags (wk1_gt3km, dk1_gt10km, ...) are False for SALs with
# no pharmacy reachable within 50 km, so they undercount; the typologies do not.
desert_defs = pd.DataFrame({
    label: df.loc[mask].groupby("PR_NAME")[POP].sum().reindex(PROVINCES, fill_value=0)
    for label, mask in definitions.items()
}).T
desert_defs["Total"] = desert_defs[PROVINCES].sum(axis=1)
desert_defs["Share of Both Provinces"] = desert_defs["Total"] / df[POP].sum()
desert_defs = desert_defs.sort_values("Total")

display(desert_defs.assign(**{c: desert_defs[c].map(millions) for c in [*PROVINCES, "Total"]},
                         **{"Share of Both Provinces": desert_defs["Share of Both Provinces"].map("{:.0%}".format)}))

fig, ax = plt.subplots(figsize=(12, 5.5))
left = np.zeros(len(desert_defs))
for prov in PROVINCES:
    ax.barh(desert_defs.index, desert_defs[prov], left=left, height=0.7, color=PROV_COLORS[prov], label=prov)
    left += desert_defs[prov].to_numpy()
for y, total in enumerate(desert_defs["Total"]):
    ax.text(total, y, f"  {total / 1e6:.1f}M", va="center", fontsize=9, color=INK)
ax.axvline(HEADLINE, color=INK, ls="--", lw=1, label="Headline: 5M")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(millions))
ax.set_xlabel("Residents (Modeled Population, 2023)")
title(fig, "How Many People Live in a Pharmacy Desert? It Depends on the Definition")
ax.legend(loc="lower right")
style_bar_axis(ax, "x")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("A1 desert population by definition")

In [ ]:
# A2. Sensitivity: residents whose nearest pharmacy is farther than X km
thresholds = np.arange(0, 15.25, 0.25)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, (mode, col, cutoff) in zip(axes, [("Walking", "wlk_k1_km", 3), ("Driving", "drv_k1_km", 10)]):
    dist = df[col].fillna(np.inf)  # unreachable within 50 km counts as beyond every threshold
    for prov in PROVINCES:
        in_prov = df["PR_NAME"].eq(prov)
        ax.plot(thresholds, [df.loc[in_prov & (dist > t), POP].sum() for t in thresholds],
                color=PROV_COLORS[prov], lw=2, label=prov)
    ax.plot(thresholds, [df.loc[dist > t, POP].sum() for t in thresholds],
            color=INK, lw=2.5, label="Both Provinces")
    ax.axhline(HEADLINE, color=INK, ls="--", lw=1)
    ax.axvline(cutoff, color=INK, ls=":", lw=1.2)
    ax.text(cutoff, ax.get_ylim()[1] * 0.95, f" Typology Threshold ({cutoff} km)", fontsize=8, color=INK)
    ax.set_title(f"{mode} Network")
    ax.set_xlabel("Distance to Nearest Pharmacy (km)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(millions))
    style_bar_axis(ax, "y")
axes[0].set_ylabel("Residents Beyond This Distance")
axes[0].legend()
title(fig, "Population Beyond Each Distance Threshold")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("A2 distance threshold sensitivity")

In [ ]:
# A3. Robustness: how much of the desert population sits in snap-flagged SALs?
for label, flag, snap in [("Walk", "walk_desert", "wlk_snapfl"), ("Drive", "drive_desert", "drv_snapfl")]:
    total = df.loc[df[flag], POP].sum()
    flagged = df.loc[df[flag] & df[snap], POP].sum()
    overall = df.loc[df[snap], POP].sum() / df[POP].sum()
    print(f"{label} desert: {total / 1e6:.2f}M residents; {flagged / 1e6:.2f}M ({flagged / total:.0%}) "
          f"are in snap-flagged SALs (vs {overall:.0%} of all residents). "
          f"Excluding them leaves {(total - flagged) / 1e6:.2f}M.")

## B. Where the Deserts Are

Maps can take a minute each with ~38,000 polygons.


In [ ]:
# MAP B1. Walking access typology
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
for ax, prov in zip(axes, PROVINCES):
    plot_categories(ax, gdf[gdf["PR_NAME"] == prov], "wlk_type", TYPE6_COLORS)
    ax.set_title(prov)
legend_below(fig, {k: v for k, v in TYPE6_COLORS.items() if k not in ("Access Gap", "Artifact Zone")}, ncol=4)
fig.text(0.5, -0.049, "Access Gap and Artifact Zone: No SALs in Current Data", ha="center", fontsize=8, color=INK)
title(fig, "Walking Access Typology")
plt.tight_layout(rect=[0, 0.04, 1, 0.95])
save_fig("B1 map walking typology")

In [ ]:
# MAP B2. Apartheid footprint vs walking distance, side by side
fig, axes = plt.subplots(2, 2, figsize=(15, 15))
for row, prov in zip(axes, PROVINCES):
    sub = gdf[gdf["PR_NAME"] == prov]
    plot_categories(row[0], sub, "settlement", SETTLEMENT_COLORS)
    row[0].set_title(f"{prov}: Settlement Type")
    plot_categories(row[1], sub, "walk_dist_class", DIST_COLORS)
    row[1].set_title(f"{prov}: Walk Distance to Nearest Pharmacy")
legend_below(fig, SETTLEMENT_COLORS, ncol=4, title="Settlement Type", y=0.035)
legend_below(fig, DIST_COLORS, ncol=7, title="Walk Distance to Nearest Pharmacy", y=-0.01)
title(fig, "Settlement Type and Walking Distance to the Nearest Pharmacy")
plt.tight_layout(rect=[0, 0.07, 1, 0.95])
save_fig("B2 map settlement vs walk distance")

In [ ]:
# MAP B3. Share Black African (2011) with walking pharmacy deserts outlined
pct_bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0001]
pct_labels = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]
pct_colors = dict(zip(pct_labels, ramp_colors(COUNT_CMAP, 5)))
gdf["pct_blk_class"] = pd.cut(gdf["pct_blk_afr"], pct_bins, labels=pct_labels, right=False).astype("object")

fig, axes = plt.subplots(1, 2, figsize=(15, 8))
for ax, prov in zip(axes, PROVINCES):
    sub = gdf[gdf["PR_NAME"] == prov]
    plot_categories(ax, sub, "pct_blk_class", pct_colors)
    deserts = sub[sub["walk_desert"]].dissolve()
    deserts.boundary.plot(ax=ax, color=VERMILLION, linewidth=1.7)
    ax.set_title(prov)
legend_below(fig, {**pct_colors, "No Population Group Data": MISSING_COLOR}, ncol=6, title="Black African Share of 2011 Population (2011 Census)")
fig.legend(handles=[plt.Line2D([], [], color=VERMILLION, label="Walking Pharmacy Desert")], loc="upper right")
title(fig, "Black African Share of Residents and Walking Pharmacy Deserts")
plt.tight_layout(rect=[0, 0.08, 1, 0.95])
save_fig("B3 map black african share walk deserts")

## C. Settlement Type and Wealth Class

Settlement Types follow DAIR's classes, which separate townships from suburbs. "Traditional" areas are largely former homelands under traditional authorities.


In [ ]:
# C1. Share of residents living in a pharmacy desert, by settlement type
groups = [g for g in SETTLEMENT_ORDER if g != NO_DWELLINGS]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (label, flag) in zip(axes, [("Walking", "walk_desert"), ("Driving", "drive_desert")]):
    table = share_table(df, ["settlement", "PR_NAME"], flag).unstack("PR_NAME").reindex(groups)[PROVINCES]
    table.iloc[::-1].plot.barh(ax=ax, color=[PROV_COLORS[p] for p in PROVINCES], width=0.75)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1))
    ax.set_title(f"{label} Pharmacy Desert")
    ax.set_xlabel("Share of Residents")
    ax.set_ylabel("")
    style_bar_axis(ax, "x")
    ax.legend()
title(fig, "Who Lives in Pharmacy Deserts, by Settlement Type")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("C1 desert share by settlement")

In [ ]:
# C2. Same question by DAIR Wealth Class
econ_order = WEALTH_ORDER
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (label, flag) in zip(axes, [("Walking", "walk_desert"), ("Driving", "drive_desert")]):
    table = share_table(df, ["econ_stat", "PR_NAME"], flag).unstack("PR_NAME").reindex(econ_order)[PROVINCES]
    table.index = [WEALTH_LABELS[k] for k in table.index]
    table.plot.bar(ax=ax, color=[PROV_COLORS[p] for p in PROVINCES], rot=0)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
    bar_labels(ax)
    style_bar_axis(ax, "y")
    ax.set_title(f"{label} Pharmacy Desert")
    ax.set_xlabel("")
    ax.legend()
axes[0].set_ylabel("Share of Residents")
title(fig, "Who Lives in Pharmacy Deserts, by Wealth Class")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("C2 desert share by wealth class")

In [ ]:
# C3. Where the desert population lives (absolute numbers, both provinces)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, (label, flag) in zip(axes, [("Walking", "walk_desert"), ("Driving", "drive_desert")]):
    counts = (df[df[flag]].groupby(["settlement", "PR_NAME"])[POP].sum()
              .unstack("PR_NAME").reindex(index=groups, columns=PROVINCES).fillna(0))
    counts.iloc[::-1].plot.barh(ax=ax, stacked=True, color=[PROV_COLORS[p] for p in PROVINCES])
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(millions))
    style_bar_axis(ax, "x")
    ax.set_title(f"{label} Desert Residents: {counts.to_numpy().sum() / 1e6:.1f}M")
    ax.set_ylabel("")
    ax.set_xlabel("Residents (Modeled Population, 2023)")
    ax.legend()
title(fig, "Where Pharmacy Desert Residents Live, by Settlement Type")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("C3 desert residents by settlement")

In [ ]:
# C4. Distribution of walking distance by settlement type (each SAL counts once)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
for ax, prov in zip(axes, PROVINCES):
    sub = df[df["PR_NAME"].eq(prov) & df["wlk_k1_km"].gt(0)]
    present = [g for g in groups if (sub["settlement"] == g).sum() >= 20]
    data = [sub.loc[sub["settlement"] == g, "wlk_k1_km"] for g in present]
    box = ax.boxplot(data, orientation="horizontal", showfliers=False, patch_artist=True, widths=0.6, medianprops={"color": INK, "linewidth": 1.5}, whiskerprops={"color": INK}, capprops={"color": INK})
    for patch, median, g in zip(box["boxes"], box["medians"], present):
        patch.set_facecolor(SETTLEMENT_COLORS[g])
        patch.set_edgecolor(HEAVY_BEIGE)
        median.set_color(contrast_text(SETTLEMENT_COLORS[g]))
    ax.set_yticks(range(1, len(present) + 1),
                  [f"{g} (n={len(d):,})" for g, d in zip(present, data)])
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%g"))
    ax.axvline(3, color=INK, ls="--", lw=1)
    ax.set_title(prov)
    ax.set_xlabel("Walk Distance to Nearest Pharmacy (km, Log Scale)")
title(fig, "Walking Distance by Settlement Type (SALs Reachable Within 50 km)")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("C4 walk distance by settlement")

In [ ]:
# C5. Heat matrix: how each settlement type's residents split across the walking typology
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, prov in zip(axes, PROVINCES):
    sub = df[df["PR_NAME"] == prov]
    table = (sub.pivot_table(index="settlement", columns="wlk_type", values=POP, aggfunc="sum")
             .reindex(index=groups, columns=list(TYPE6_COLORS)))
    table = table.div(table.sum(axis=1), axis=0)
    heat_matrix(ax, table, cmap=SHARE_CMAP, vmin=0, vmax=1, title=prov)
title(fig, "Share of Each Settlement Type's Residents in Each Walking Access Typology Class")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("C5 settlement by walking typology")

## D. Population Group (2011 Census)

Shares use 2011 Population Group counts per SAL, combined with current access measures.


In [ ]:
# D1. Exposure: share of each group's residents living in a pharmacy desert
rows = []
for prov in PROVINCES:
    sub = df[df["PR_NAME"] == prov]
    for col, group in RACE_COLS.items():
        people = sub[col].fillna(0)
        rows.append({
            "province": prov, "group": group, "residents_2011": people.sum(),
            "walk_desert_share": people[sub["walk_desert"]].sum() / people.sum(),
            "drive_desert_share": people[sub["drive_desert"]].sum() / people.sum(),
            "no_pharmacy_within_50km_walk": people[sub["wlk_k1_km"].isna()].sum() / people.sum(),
            # unreachable residents are counted at the 50 km cap
            "median_walk_km": weighted_median(sub["wlk_k1_km"].fillna(50).clip(upper=50), people),
        })
exposure = pd.DataFrame(rows)
display(exposure.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, col, label in zip(axes, ["walk_desert_share", "drive_desert_share"], ["Walking", "Driving"]):
    table = exposure.pivot(index="group", columns="province", values=col).reindex(list(RACE_COLORS))
    table[PROVINCES].plot.bar(ax=ax, color=[PROV_COLORS[p] for p in PROVINCES], rot=0)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
    bar_labels(ax)
    style_bar_axis(ax, "y")
    ax.set_title(f"{label} Pharmacy Desert")
    ax.set_xlabel("")
    ax.legend()
axes[0].set_ylabel("Share of Group's Residents (2011 Census)")
title(fig, "Share of Each Population Group Living in a Pharmacy Desert")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("D1 desert share by population group")

In [ ]:
# D2. Who lives inside vs outside walking pharmacy deserts
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for row, prov in zip(axes, PROVINCES):
    sub = df[df["PR_NAME"] == prov]
    for ax, (label, mask) in zip(row, [("In a Walking Desert", sub["walk_desert"]),
                                        ("Not in a Walking Desert", ~sub["walk_desert"])]):
        totals = sub.loc[mask, list(RACE_COLS)].sum().rename(RACE_COLS)
        ax.pie(totals, colors=[RACE_COLORS[g] for g in totals.index], startangle=90,
               counterclock=False, wedgeprops={"edgecolor": OFF_WHITE, "linewidth": 1.5}, textprops={"color": OFF_WHITE, "fontsize": 9, "fontweight": "semibold"},
               autopct=lambda p: f"{p:.0f}%" if p >= 4 else "", pctdistance=0.75)
        ax.set_title(f"{prov}\n{label} ({totals.sum() / 1e6:.1f}M, 2011 Census)", fontsize=10)
legend_below(fig, RACE_COLORS, ncol=5, title="Population Group (2011 Census)")
title(fig, "Population Groups Inside and Outside Walking Pharmacy Deserts")
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
save_fig("D2 population groups in vs out of deserts")

In [ ]:
# D3. Black African share vs walking distance (hexbin) with population-weighted median
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
bins = np.linspace(0, 1, 11)
for ax, prov in zip(axes, PROVINCES):
    in_prov = df["PR_NAME"].eq(prov) & df["pct_blk_afr"].notna()
    n_unreachable = (in_prov & df["wlk_k1_km"].isna()).sum()
    sub = df[in_prov & df["wlk_k1_km"].gt(0)]
    hb = ax.hexbin(sub["pct_blk_afr"], sub["wlk_k1_km"], gridsize=35, yscale="log",
                   bins="log", cmap="dair_count", mincnt=1)
    mids, medians = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        in_bin = sub[(sub["pct_blk_afr"] >= lo) & (sub["pct_blk_afr"] < hi + (hi == 1))]
        mids.append((lo + hi) / 2)
        medians.append(weighted_median(in_bin["wlk_k1_km"], in_bin[POP]))
    ax.plot(mids, medians, color=VERMILLION, lw=2.5, marker="o", label="Population-Weighted Median")
    # Spearman's rho as Pearson on ranks; pandas' Spearman method imports scipy, which is broken in the za env.
    rho = sub["pct_blk_afr"].rank().corr(sub["wlk_k1_km"].rank())
    ax.set_title(f"{prov}: Spearman's Rho = {rho:.2f}\n{len(sub):,} SALs; "
                 f"{n_unreachable:,} With No Pharmacy Within 50 km Not Shown", fontsize=10)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%g"))
    ax.set_xlabel("Black African Share of 2011 Population")
    ax.legend(loc="upper left")
    cb = fig.colorbar(hb, ax=ax, label="SALs (Log Scale)")
    cb.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%g"))
    cb.ax.minorticks_off()
axes[0].set_ylabel("Walk Distance to Nearest Pharmacy (km, Log Scale)")
title(fig, "Black African Share vs. Walking Distance to the Nearest Pharmacy")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("D3 black african share vs walk distance")

In [ ]:
# D4. Is it Population Group or rurality? Desert share by Majority Population Group within each geography type
min_pop = 20_000  # hide cells with too few residents to be meaningful
gtype_order = ["Urban", "Traditional", "Farms"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, prov in zip(axes, PROVINCES):
    sub = df[df["PR_NAME"] == prov]
    table = (share_table(sub, ["EA_GTYPE", "majority_group"], "walk_desert", min_pop=min_pop)
             .unstack("majority_group")
             .reindex(index=gtype_order, columns=[g for g in RACE_COLORS if g != "Other"]))
    heat_matrix(ax, table, cmap=SHARE_CMAP, vmin=0, vmax=1, title=prov)
    ax.set_xlabel("Majority Population Group of SAL (2011 Census)")
    ax.set_ylabel("Geography Type")
title(fig, f"Share of Residents in a Walking Desert (Blank = Under {min_pop:,} Residents)")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("D4 desert share by geography and majority group")

## E. Walking vs. Driving: Access That Depends on Vehicle Ownership


In [ ]:
# E1. Residents by walking typology x driving typology
order = list(TYPE6_COLORS)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, prov in zip(axes, PROVINCES):
    sub = df[df["PR_NAME"] == prov]
    table = (sub.pivot_table(index="wlk_type", columns="drv_type", values=POP, aggfunc="sum")
             .reindex(index=order, columns=order) / 1e6)
    heat_matrix(ax, table, fmt="{:.2f}", cmap=COUNT_CMAP, title=f"{prov} (Millions of Residents)")
    ax.set_xlabel("Driving Access Typology")
    ax.set_ylabel("Walking Access Typology")
title(fig, "Residents by Walking and Driving Access Typology")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("E1 walking vs driving typology")

walk_deserts = df[df["walk_desert"]].assign(car_ok=lambda d: d["drv_type"].eq("Well-Served"))
print(f"Walking Pharmacy Desert but Well-Served by Car: {walk_deserts.loc[walk_deserts['car_ok'], POP].sum() / 1e6:.2f}M residents")
display(share_table(walk_deserts, ["settlement", "PR_NAME"], "car_ok").unstack("PR_NAME")
        .reindex(groups).round(2)
        .rename_axis("Share of Walking-Desert Residents Well-Served by Car"))

## F. Distributions Behind the Scores


In [ ]:
# F1. Zero-inflation of Ai, and the spread of nonzero scores
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for row, (mode, col) in zip(axes, [("Walk", "wlk_ai"), ("Drive", "drv_ai")]):
    for ax, prov in zip(row, PROVINCES):
        vals = df.loc[df["PR_NAME"] == prov, col]
        zero_share = vals.eq(0).mean()
        ax.hist(np.log1p(vals[vals > 0]), bins=50, color=PROV_COLORS[prov], alpha=0.9, edgecolor=OFF_WHITE, linewidth=0.4)
        ax.set_title(f"{prov}, {mode} Access Score: {zero_share:.0%} of SALs Are Exactly 0")
        ax.set_xlabel(f"ln(1 + {col}) for Nonzero SALs")
        ax.set_ylabel("SALs")
        style_bar_axis(ax, "y")
title(fig, "Access Scores Are Zero-Inflated")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("F1 ai zero inflation")

In [ ]:
# F2. Residents by walking distance to the nearest pharmacy (population-weighted)
fig, ax = plt.subplots(figsize=(11, 4.5))
log_bins = np.logspace(-2, np.log10(50), 60)
for prov in PROVINCES:
    sub = df[df["PR_NAME"].eq(prov) & df["wlk_k1_km"].gt(0)]
    ax.hist(sub["wlk_k1_km"], bins=log_bins, weights=sub[POP], histtype="step", lw=2,
            color=PROV_COLORS[prov], label=prov)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%g"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(millions))
ax.axvline(3, color=INK, ls="--", lw=1)
ax.set_xlabel("Walk Distance to Nearest Pharmacy (km, Log Scale)")
ax.set_ylabel("Residents (Modeled Population, 2023)")
title(fig, "Residents by Walking Distance to the Nearest Pharmacy")
ax.legend()
style_bar_axis(ax, "y")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig("F2 residents by walk distance")

## G. Numbers for the Write-Up


In [ ]:
def fmt_share(frame, by, value, flag):
    return share_table(frame, by, flag).get(value, np.nan)

walk_total = df.loc[df["walk_desert"], POP].sum()
drive_total = df.loc[df["drive_desert"], POP].sum()
print(f"Residents, both provinces (Modeled Population, 2023): {df[POP].sum() / 1e6:.1f}M")
print(f"Walking Pharmacy Desert:  {walk_total / 1e6:.2f}M ({walk_total / df[POP].sum():.0%})")
print(f"Driving Pharmacy Desert:  {drive_total / 1e6:.2f}M ({drive_total / df[POP].sum():.0%})")
for prov in PROVINCES:
    sub = df[df["PR_NAME"] == prov]
    print(f"\n{prov}")
    for g in ["Suburb", "Township", "Informal Settlement", "Traditional (Former Homeland)"]:
        print(f"  {g:32s} walking desert share: {fmt_share(sub, 'settlement', g, 'walk_desert'):.0%}")
    for g in ["Black African", "White"]:
        share = exposure.query("province == @prov and group == @g")["walk_desert_share"].iat[0]
        print(f"  {g + ' residents (2011)':32s} walking desert share: {share:.0%}")

## H. Access Share

A group's **Access Share** is its share of the province's total access: the sum of each SAL's access score x Modeled Population (2023) over the group, divided by the same sum over the province, per mode. It is read next to the group's **Population Share** (its share of the province's residents). A group whose Access Share is above its Population Share has more than its share of pharmacy access.

Groups are Settlement Type (H1) and Majority Population Group (H2). For H2, SALs with no Population Group data are excluded from both numerator and denominator, as in `scripts/derive_data.py`.


In [ ]:
# H1. Access Share vs Population Share by Settlement Type, per province and mode.
# Same definition as print_access_share() in scripts/derive_data.py.
MODES = [("Walking", "wlk_ai"), ("Driving", "drv_ai")]


def access_share_table(frame, group_col, groups):
    """Access Share and Population Share for each group, per province and mode."""
    rows = []
    for prov in PROVINCES:
        sub = frame[frame["PR_NAME"].eq(prov)]
        for mode, ai_col in MODES:
            weighted = sub[ai_col] * sub[POP]
            for group in groups:
                in_group = sub[group_col].eq(group)
                rows.append({
                    "province": prov, "mode": mode, "group": group,
                    "access_share": weighted[in_group].sum() / weighted.sum(),
                    "population_share": sub.loc[in_group, POP].sum() / sub[POP].sum(),
                })
    return pd.DataFrame(rows)


def plot_access_share(table, groups, fig_title):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
    y = np.arange(len(groups))
    for row, prov in zip(axes, PROVINCES):
        for ax, (mode, _) in zip(row, MODES):
            sub = table[table["province"].eq(prov) & table["mode"].eq(mode)].set_index("group").reindex(groups)
            ax.barh(y + 0.2, sub["population_share"], height=0.4, color=HEAVY_BEIGE, label="Population Share")
            ax.barh(y - 0.2, sub["access_share"], height=0.4, color=PROV_COLORS[prov], label="Access Share")
            for yi, v in zip(y - 0.2, sub["access_share"]):
                ax.text(v, yi, f" {v:.0%}", va="center", fontsize=8, color=INK)
            ax.set_yticks(y, groups)
            ax.invert_yaxis()
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(1))
            style_bar_axis(ax, "x")
            ax.set_title(f"{prov}: {mode}")
    handles = [Patch(facecolor=PROV_COLORS["Gauteng"], label="Access Share (Gauteng)"),
               Patch(facecolor=PROV_COLORS["KwaZulu-Natal"], label="Access Share (KwaZulu-Natal)"),
               Patch(facecolor=HEAVY_BEIGE, label="Population Share")]
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=3)
    title(fig, fig_title)
    plt.tight_layout(rect=[0, 0.02, 1, 0.95])
    return fig


settlement_groups = [g for g in SETTLEMENT_ORDER if g != NO_DWELLINGS]
access_by_settlement = access_share_table(df, "settlement", settlement_groups)
display(access_by_settlement.pivot_table(index="group", columns=["province", "mode"],
                                         values=["access_share", "population_share"])
        .reindex(settlement_groups).round(3))
plot_access_share(access_by_settlement, settlement_groups, "Access Share vs. Population Share by Settlement Type")
save_fig("H1 access share by settlement type")


In [ ]:
# H2. Access Share vs Population Share by Majority Population Group.
# Population Groups are 2011 Census counts; access scores are weighted by Modeled Population (2023).
# SALs with no Population Group data are excluded, as in scripts/derive_data.py.
group_order = list(RACE_COLORS)
with_groups = df[df["majority_group"].notna()]
access_by_group = access_share_table(with_groups, "majority_group", group_order)
display(access_by_group.pivot_table(index="group", columns=["province", "mode"],
                                    values=["access_share", "population_share"])
        .reindex(group_order).round(3))
plot_access_share(access_by_group, group_order,
                  "Access Share vs. Population Share by Majority Population Group (2011 Census)")
save_fig("H2 access share by majority population group")


In [ ]:
# H3. Numbers for the write-up, formatted like derive_data.py's STAT lines.
for table in (access_by_settlement, access_by_group):
    for r in table.itertuples():
        mode = "walk" if r.mode == "Walking" else "drive"
        print(f"STAT access_share {mode} {r.province} {r.group} = "
              f"{r.access_share * 100:.1f}% (pop share {r.population_share * 100:.1f}%)")
